## Setup start 

In [ ]:
# Parameters
SNT_ROOT_PATH   <- '~/workspace'   # SNT root

In [ ]:
# Set project folders
CODE_PATH      <- file.path(SNT_ROOT_PATH, "code")
CONFIG_PATH    <- file.path(SNT_ROOT_PATH, "configuration")
FORMATTED_DATA_PATH <- file.path(SNT_ROOT_PATH, "data", "dhis2", "extracts_formatted")

# Load functions
source(file.path(CODE_PATH, "snt_utils.r"))

# Load libraries
required_packages <- c("lubridate", "zoo", "arrow", "dplyr", "tidyr", "stringr", "stringi", "jsonlite", "httr", "reticulate", "digest")
install_and_load(required_packages)

# Load openhexa.sdk 
Sys.setenv(RETICULATE_PYTHON = "/opt/conda/bin/python")
reticulate::py_config()$python
openhexa <- import("openhexa.sdk")

### Load SNT configuration


In [ ]:
# Load SNT config
config_json <- tryCatch({
        fromJSON(file.path(CONFIG_PATH, "SNT_config.json"))
    },
    error = function(e) {
        msg <- paste0("Error while loading configuration", conditionMessage(e))  
        cat(msg)   
        stop(msg) 
    })

# Save this country code in a variable
COUNTRY_CODE <- config_json$SNT_CONFIG$COUNTRY_CODE
ADMIN_2 <- toupper(config_json$SNT_CONFIG$DHIS2_ADMINISTRATION_2)

# print(config.json$SNT_CONFIG)
msg <- paste0("SNT configuration loaded from  : ", file.path(CONFIG_PATH, "SNT_config.json"))
log_msg(msg)

### Load DHIS2 pyramid data  

-Load DHIS2 pyramid from latest dataset version

In [ ]:
# DHIS2 Dataset extract identifier
dataset_name <- config_json$SNT_DATASET_IDENTIFIERS$DHIS2_DATASET_EXTRACTS

# Load file from dataset
dhis2_data <- tryCatch({ get_latest_dataset_file_in_memory(dataset_name, paste0(COUNTRY_CODE, "_dhis2_raw_pyramid.parquet")) },
                  error = function(e) {
                      msg <- paste("Error while loading DHIS2 pyramid file for: " , COUNTRY_CODE, conditionMessage(e))  # log error message
                      cat(msg)
                      stop(msg)
})

msg <- paste0("DHIS2 pyramid data loaded from dataset : ", dataset_name, " dataframe dimensions: ", paste(dim(dhis2_data), collapse=", "))
log_msg(msg)

In [ ]:
head(dhis2_data, 3)

## SNT pyramid formatting

In [ ]:
# Set value
pyramid_data <- dhis2_data

# remove columns with only NA values
pyramid_data <- pyramid_data[, colSums(!is.na(pyramid_data)) > 0]

name_columns <- colnames(pyramid_data)[grepl("_NAME", colnames(pyramid_data))]
for (column in name_columns){
    print(paste0("Format : ", column))
    # Clean strings 
    pyramid_data[[column]] <- format_names(pyramid_data[[column]])
}

# Column names to upper case
colnames(pyramid_data) <- clean_column_names(pyramid_data)
head(pyramid_data, 3)

### Extract longitude/latitude from geometry column (geoJson)

In [ ]:
# Extract longitude/latitude from GEOMETRY (geoJson) with robust parsing
extract_geo_coords <- function(geom_json) {
  if (is.na(geom_json) || !nzchar(geom_json)) {
    return(list(lon = NA_real_, lat = NA_real_, parse_ok = FALSE))
  }

  parsed <- tryCatch(jsonlite::fromJSON(geom_json), error = function(e) NULL)
  coords <- parsed$coordinates

  if (is.null(coords) || length(coords) < 2) {
    return(list(lon = NA_real_, lat = NA_real_, parse_ok = FALSE))
  }

  list(
    lon = suppressWarnings(as.numeric(coords[[1]])),
    lat = suppressWarnings(as.numeric(coords[[2]])),
    parse_ok = TRUE
  )
}

shift_decimal_left_to_right <- function(value, k) {
  if (is.na(value)) return(NA_real_)

  sign_val <- ifelse(value < 0, -1, 1)
  s <- gsub("[^0-9]", "", as.character(abs(value)))
  if (!nzchar(s) || nchar(s) <= k) return(NA_real_)

  shifted <- paste0(substr(s, 1, k), ".", substr(s, k + 1, nchar(s)))
  sign_val * suppressWarnings(as.numeric(shifted))
}

inside_bdi_bbox <- function(lon, lat) {
  !is.na(lon) && !is.na(lat) &&
    abs(lon) <= 180 && abs(lat) <= 90 &&
    dplyr::between(lon, 28.9, 30.9) &&
    dplyr::between(lat, -4.8, -2.1)
}

build_candidates <- function(lon, lat, max_shift = 2) {
  candidates <- list(
    ORIGINAL = c(lon, lat),
    SWAP = c(lat, lon),
    FLIP_LAT = c(lon, -lat),
    FLIP_LON = c(-lon, lat),
    FLIP_BOTH = c(-lon, -lat),
    SWAP_FLIP_LAT = c(lat, -lon),
    SWAP_FLIP_LON = c(-lat, lon),
    SWAP_FLIP_BOTH = c(-lat, -lon)
  )

  for (k in seq_len(max_shift)) {
    lon_lr <- shift_decimal_left_to_right(lon, k)
    lat_lr <- shift_decimal_left_to_right(lat, k)

    candidates[[paste0("DECIMAL_LR_LON_", k)]] <- c(lon_lr, lat)
    candidates[[paste0("DECIMAL_LR_LAT_", k)]] <- c(lon, lat_lr)
    candidates[[paste0("DECIMAL_LR_BOTH_", k)]] <- c(lon_lr, lat_lr)

    candidates[[paste0("SWAP_DECIMAL_LR_LON_", k)]] <- c(lat_lr, lon)
    candidates[[paste0("SWAP_DECIMAL_LR_LAT_", k)]] <- c(lat, lon_lr)
    candidates[[paste0("SWAP_DECIMAL_LR_BOTH_", k)]] <- c(lat_lr, lon_lr)
  }

  candidates
}

fix_coordinate_pair_bbox <- function(lon, lat, max_shift = 2) {
  if (is.na(lon) || is.na(lat)) {
    return(list(LONGITUDE = NA_real_, LATITUDE = NA_real_, METHOD = "MISSING_COORDINATES", VALID = FALSE))
  }

  candidates <- build_candidates(lon, lat, max_shift = max_shift)
  method_names <- names(candidates)

  for (method in method_names) {
    cand <- candidates[[method]]
    lon_c <- as.numeric(cand[1])
    lat_c <- as.numeric(cand[2])
    if (inside_bdi_bbox(lon_c, lat_c)) {
      return(list(LONGITUDE = lon_c, LATITUDE = lat_c, METHOD = method, VALID = TRUE))
    }
  }

  return(list(LONGITUDE = NA_real_, LATITUDE = NA_real_, METHOD = "INVALID_NO_MATCH", VALID = FALSE))
}

coords_list <- lapply(pyramid_data$GEOMETRY, extract_geo_coords)
coords_df <- dplyr::bind_rows(lapply(coords_list, as.data.frame))

pyramid_data <- pyramid_data %>%
  mutate(
    LONGITUDE = coords_df$lon,
    LATITUDE = coords_df$lat
  ) %>%
  select(-GEOMETRY)

lon0 <- pyramid_data$LONGITUDE
lat0 <- pyramid_data$LATITUDE

coord_fix_df <- dplyr::tibble(
  LONGITUDE_ORIGINAL = lon0,
  LATITUDE_ORIGINAL = lat0,
  LONGITUDE_FIXED = NA_real_,
  LATITUDE_FIXED = NA_real_,
  COORD_FIX_METHOD = NA_character_,
  COORD_IS_VALID = FALSE
)

fix_results <- lapply(seq_along(lon0), function(i) {
  fix_coordinate_pair_bbox(lon0[i], lat0[i], max_shift = 2)
})

for (i in seq_along(fix_results)) {
  fr <- fix_results[[i]]
  coord_fix_df$LONGITUDE_FIXED[i] <- fr$LONGITUDE
  coord_fix_df$LATITUDE_FIXED[i] <- fr$LATITUDE
  coord_fix_df$COORD_FIX_METHOD[i] <- fr$METHOD
  coord_fix_df$COORD_IS_VALID[i] <- fr$VALID
}

pyramid_data$LONGITUDE <- coord_fix_df$LONGITUDE_FIXED
pyramid_data$LATITUDE <- coord_fix_df$LATITUDE_FIXED

invalid_coords <- pyramid_data %>%
  bind_cols(coord_fix_df %>% dplyr::select(LONGITUDE_ORIGINAL, LATITUDE_ORIGINAL, COORD_FIX_METHOD, COORD_IS_VALID)) %>%
  dplyr::filter(!COORD_IS_VALID & !is.na(LONGITUDE_ORIGINAL) & !is.na(LATITUDE_ORIGINAL)) %>%
  dplyr::mutate(INVALID_COORD_REASON = "NO_VALID_TRANSFORMATION_IN_BDI_BBOX")

n_total_coords <- sum(!is.na(coord_fix_df$LONGITUDE_ORIGINAL) & !is.na(coord_fix_df$LATITUDE_ORIGINAL))
n_kept_original <- sum(coord_fix_df$COORD_FIX_METHOD == "ORIGINAL", na.rm = TRUE)
n_corrected <- sum(coord_fix_df$COORD_IS_VALID & coord_fix_df$COORD_FIX_METHOD != "ORIGINAL", na.rm = TRUE)
n_invalid <- nrow(invalid_coords)

log_msg(glue::glue("Coordinate quality check over {n_total_coords} FOSAs: original valid={n_kept_original}, corrected={n_corrected}, invalid={n_invalid}."))
if (n_corrected > 0) {
  log_msg(glue::glue("[WARNING] Applied coordinate correction algorithm to {n_corrected} FOSAs (swap/sign/decimal left-to-right, k<=2)."), "warning")
}
if (n_invalid > 0) {
  log_msg(glue::glue("[WARNING] {n_invalid} FOSAs remain invalid after all correction attempts. LONGITUDE/LATITUDE set to NA."), "warning")
}

parse_failures <- sum(!coords_df$parse_ok, na.rm = TRUE)
if (parse_failures > 0) {
  log_msg(glue::glue("[WARNING] {parse_failures} GEOMETRY records could not be parsed. Resulting LONGITUDE/LATITUDE are NA for those rows."), "warning")
}

head(pyramid_data, 3)

### (!) BDI specific mapping (start)
  
-Load and apply mappings at Province and Commune levels.  
-We **CREATE NEW IDs** so the aggregations can work following the new mappings.

In [ ]:
# Select valid districts
# pyramid_data <- pyramid_data[grepl('^DS', pyramid_data$LEVEL_3_NAME), ]

In [ ]:
# NOTE: double check cases like lvl 6 id = "ncaS81uNi5d".

In [ ]:
# Retrieve the missing ids 
# unmapped_level6_ids <- setdiff(unique(pyramid_data$LEVEL_6_ID), unique(mappings$LEVEL_6_ID))

In [ ]:
# Load mappings (BDI)
mappings <- tryCatch({ read_parquet(file.path(SNT_ROOT_PATH, "pipelines/snt_dhis2_formatting/dev/inputs/BDI_DHIS2_FOSAs_cleaned_matched.parquet")) },
    error = function(e) {
        stop("Can't find the BDI org unit mappings file!")
    }
)

In [ ]:
# 1. Collapse at these levels 
commune_mapping <- mappings %>%
  group_by(PROVINCE_NAME_OLD, DISTRICT_NAME_OLD, COMMUNE_NAME_OLD) %>%
  summarise(
    COMMUNE_NAME_NEW = first(COMMUNE_NAME_NEW),
    PROVINCE_NAME_NEW = first(PROVINCE_NAME_NEW),
    .groups = "drop"
  )

# 2. Match mappings
pyramid_with_commune_new <- pyramid_data %>%
  left_join(
    commune_mapping,
    by = c(
      "LEVEL_2_NAME" = "PROVINCE_NAME_OLD",
      "LEVEL_3_NAME" = "DISTRICT_NAME_OLD",
      "LEVEL_4_NAME" = "COMMUNE_NAME_OLD"
    )
  )

# 3. HARD VALIDATION
if (nrow(pyramid_with_commune_new %>% filter(!is.na(LEVEL_2_NAME) & is.na(PROVINCE_NAME_NEW))) > 0)
  stop("Unmapped provinces detected")
if (nrow(pyramid_with_commune_new %>% filter(!is.na(LEVEL_4_NAME) & is.na(COMMUNE_NAME_NEW))) > 0)
  stop("Unmapped communes detected")

# 4. Build IDs
pyramid_with_commune_new$PROVINCE_ID_NEW <- vapply(pyramid_with_commune_new$PROVINCE_NAME_NEW, 
                                                   function(x) digest(x, algo = "xxhash64"), character(1))
pyramid_with_commune_new$COMMUNE_ID_NEW <- vapply(paste(pyramid_with_commune_new$PROVINCE_NAME_NEW, pyramid_with_commune_new$COMMUNE_NAME_NEW, sep = "||"), 
                                                  function(x) digest(x, algo = "xxhash64"), character(1))

In [ ]:
# Manual fixes:
# qvN9HxzNpJE - CDS IZERE COMMUNE_NAME_NEW -> NGOZI 496549908fbc9f65
# oqys99wvXM7 - CDS PENIEL COMMUNE_NAME_NEW -> NGOZI 
# yBWYywBS3kW	- CDS RUPARI COMMUNE_NAME_NEW -> NGOZI 

pyramid_with_commune_new$COMMUNE_NAME_NEW[pyramid_with_commune_new$LEVEL_6_ID == "qvN9HxzNpJE"] <- "NGOZI"
pyramid_with_commune_new$COMMUNE_ID_NEW[pyramid_with_commune_new$LEVEL_6_ID == "qvN9HxzNpJE"]   <- "496549908fbc9f65"
pyramid_with_commune_new$COMMUNE_NAME_NEW[pyramid_with_commune_new$LEVEL_6_ID == "oqys99wvXM7"] <- "NGOZI"
pyramid_with_commune_new$COMMUNE_ID_NEW[pyramid_with_commune_new$LEVEL_6_ID == "oqys99wvXM7"]   <- "496549908fbc9f65"
pyramid_with_commune_new$COMMUNE_NAME_NEW[pyramid_with_commune_new$LEVEL_6_ID == "yBWYywBS3kW"] <- "NGOZI"
pyramid_with_commune_new$COMMUNE_ID_NEW[pyramid_with_commune_new$LEVEL_6_ID == "yBWYywBS3kW"]   <- "496549908fbc9f65"

# Missing info for :
# V6odL3D8UCX - CDS AGAPE to which commune should we set this one?

In [ ]:
# 5. Rename cols 
pyramid_data <- pyramid_with_commune_new %>%
    mutate(
        LEVEL_2_NAME = PROVINCE_NAME_NEW,
        LEVEL_4_NAME = COMMUNE_NAME_NEW,
        LEVEL_2_ID = PROVINCE_ID_NEW,
        LEVEL_4_ID = COMMUNE_ID_NEW
    ) %>%
    select(-ends_with("_NEW"), -starts_with("key_"))

head(pyramid_data, 3)

In [ ]:
# unique(pyramid_with_commune_new[pyramid_with_commune_new$PROVINCE_NAME_NEW=="BUJUMBURA", ]$COMMUNE_NAME_NEW)
# unique(pyramid_with_commune_new[pyramid_with_commune_new$PROVINCE_NAME_NEW=="BURUNGA", ]$COMMUNE_NAME_NEW) 

### (!) BDI specific mapping (end)

### Output formatted pyramid data

In [ ]:
out_msg <- paste0("Pyramid data saved under: ", file.path(FORMATTED_DATA_PATH, paste0(COUNTRY_CODE, "_pyramid.parquet")))
invalid_coords_report_path <- file.path(FORMATTED_DATA_PATH, paste0(COUNTRY_CODE, "_pyramid_invalid_coordinates.csv"))

# write parquet file
write_parquet(pyramid_data, file.path(FORMATTED_DATA_PATH, paste0(COUNTRY_CODE, "_pyramid.parquet")))

# write csv file
write.csv(pyramid_data, file.path(FORMATTED_DATA_PATH, paste0(COUNTRY_CODE, "_pyramid.csv")), row.names = FALSE)

# Write invalid coordinates report when needed
if (nrow(invalid_coords) > 0) {
    write.csv(invalid_coords, invalid_coords_report_path, row.names = FALSE)
    log_msg(glue::glue("[WARNING] Invalid coordinates report saved under: {invalid_coords_report_path}"), "warning")
}

In [ ]:
# log
log_msg(out_msg)

### Data Summary 

In [ ]:
# Data summary
print(summary(pyramid_data))